# Neural Autoregressive Language Model — Vanilla RNN

This notebook is **standalone**: it prepares data, builds a character-level language model, trains it, evaluates it, generates text, and saves a checkpoint.

## How to use
1. Optionally upload a UTF-8 text file named `corpus.txt` next to this notebook. If absent, a small built-in demo corpus is used so every cell can run.
2. Run the notebook from top to bottom.
3. For meaningful results, use a larger domain-relevant corpus. The demo corpus is only for verifying the pipeline.

## What this notebook teaches
- Autoregressive next-character prediction: predict the next character from the preceding context.
- Train/validation/test separation.
- Cross-entropy loss and character-level perplexity.
- Temperature-controlled sampling.
- Saving and reloading a trained model.

**Important:** character-level perplexity is not directly comparable to word- or token-level perplexity from other models.

## 1. Imports, reproducibility, and corpus

The built-in corpus is for pipeline verification only. Place a larger UTF-8 `corpus.txt` in the working directory for useful experiments.

In [1]:

from pathlib import Path
import math, random, json, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

CORPUS_PATH = Path("/content/corpus.txt")
DEMO_TEXT = """
Autoregressive models learn to predict the next symbol from the symbols that came before it.
A language model estimates a probability distribution over sequences of text.
At each step, the model receives a context and predicts a distribution for the next character.
Training adjusts model parameters to reduce next-character cross entropy.
A recurrent neural network maintains a hidden state that summarizes earlier input.
An LSTM uses gates and a cell state to help preserve information across longer sequences.
Neural language models can generate text by repeatedly sampling the next character.
Evaluation should use held-out text that was not used to fit the model.
Machine learning experiments should be reproducible, measurable, and resource conscious.
This example is deliberately small: use your own larger corpus for meaningful results.
"""

if CORPUS_PATH.exists():
    text = CORPUS_PATH.read_text(encoding="utf-8")
    corpus_source = str(CORPUS_PATH)
else:
    text = DEMO_TEXT.strip() + "\n"
    corpus_source = "built-in demo corpus"

text = text.replace("\r\n", "\n").replace("\r", "\n")
if len(text) < 200:
    raise ValueError("Corpus is too short. Provide at least 200 characters in corpus.txt.")

# Split the raw stream before fitting the vocabulary to reduce leakage.
n = len(text)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
train_text = text[:train_end]
val_text = text[train_end:val_end]
test_text = text[val_end:]

special_tokens = ["<unk>"]
chars = sorted(set(train_text))
itos = special_tokens + chars
stoi = {ch: i for i, ch in enumerate(itos)}
UNK_ID = stoi["<unk>"]

def encode(s):
    return [stoi.get(ch, UNK_ID) for ch in s]

train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)
test_ids = torch.tensor(encode(test_text), dtype=torch.long)

print(f"Corpus source: {corpus_source}")
print(f"Characters: {n:,} | train: {len(train_text):,} | val: {len(val_text):,} | test: {len(test_text):,}")
print(f"Vocabulary size: {len(itos)}")
if corpus_source == "built-in demo corpus":
    print("NOTE: Demo data verifies the workflow; use a larger corpus to assess language-model quality.")


Device: cuda
Corpus source: /content/corpus.txt
Characters: 10,831 | train: 8,664 | val: 1,083 | test: 1,084
Vocabulary size: 114


## 2. Create next-character training examples

Each input sequence is paired with its one-character-shifted target. The model learns next-character prediction at every sequence position.

In [2]:

BLOCK_SIZE = 32
BATCH_SIZE = 64

class NextCharDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size)

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.block_size]
        y = self.ids[idx + 1 : idx + self.block_size + 1]
        return x, y

train_ds = NextCharDataset(train_ids, BLOCK_SIZE)
val_ds = NextCharDataset(val_ids, BLOCK_SIZE)
test_ds = NextCharDataset(test_ids, BLOCK_SIZE)

if min(len(train_ds), len(val_ds), len(test_ds)) == 0:
    raise ValueError("One split is shorter than BLOCK_SIZE. Use a larger corpus or reduce BLOCK_SIZE.")

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=shuffle, drop_last=False)

train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds, shuffle=False)
test_loader = make_loader(test_ds, shuffle=False)

xb, yb = next(iter(train_loader))
print("Input batch:", tuple(xb.shape), "| Target batch:", tuple(yb.shape))
print("Example input :", repr("".join(itos[i] if i != UNK_ID else "�" for i in xb[0].tolist())))
print("Example target:", repr("".join(itos[i] if i != UNK_ID else "�" for i in yb[0].tolist())))


Input batch: (64, 32) | Target batch: (64, 32)
Example input : ' abilities by drawing natural en'
Example target: 'abilities by drawing natural ene'


## 3. Define and train the model

The recurrent network emits a next-character distribution at every sequence position. Gradient clipping helps stabilize recurrent training.

In [3]:

# Modest hyperparameters allow a CPU smoke test; use a larger corpus for meaningful learning.
EMBED_DIM = 48
HIDDEN_DIM = 96
NUM_LAYERS = 1
DROPOUT = 0.0
LEARNING_RATE = 2e-3
EPOCHS = 500
GRAD_CLIP = 1.0

class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.recurrent = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, nonlinearity='tanh', dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, idx, state=None):
        embedded = self.embedding(idx)
        output, state = self.recurrent(embedded, state)
        logits = self.lm_head(self.dropout(output))
        return logits, state

    @torch.no_grad()
    def generate(self, start_text, max_new_chars=300, temperature=0.8):
        self.eval()
        ids = encode(start_text)
        if not ids:
            ids = [stoi.get("\n", UNK_ID)]
        # Prime the recurrent state with the prompt, then sample one character at a time.
        context = torch.tensor([ids[-BLOCK_SIZE:]], dtype=torch.long, device=DEVICE)
        generated = list(ids)
        logits, state = self(context)
        for _ in range(max_new_chars):
            next_logits = logits[:, -1, :] / max(float(temperature), 1e-5)
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_id)
            next_input = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)
            logits, state = self(next_input, state)
        return "".join(itos[i] if i < len(itos) and i != UNK_ID else "�" for i in generated)

model = RNNLanguageModel(vocab_size=len(itos)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            count = y.numel()
            total_loss += loss.item() * count
            total_tokens += count
    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, math.exp(min(avg_loss, 20.0))

best_val_loss = float("inf")
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    train_loss = total_loss / max(total_tokens, 1)
    val_loss, val_ppl = evaluate(model, val_loader)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_char_perplexity": val_ppl})
    if epoch%50==0 or epoch%EPOCHS==0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val char-PPL {val_ppl:.2f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best validation checkpoint (val loss={best_val_loss:.4f}).")


Epoch 50/500 | train loss 0.3991 | val loss 3.3270 | val char-PPL 27.85
Epoch 100/500 | train loss 0.3544 | val loss 3.6036 | val char-PPL 36.73
Epoch 150/500 | train loss 0.3333 | val loss 3.7088 | val char-PPL 40.80
Epoch 200/500 | train loss 0.3228 | val loss 3.8318 | val char-PPL 46.15
Epoch 250/500 | train loss 0.3153 | val loss 3.9062 | val char-PPL 49.71
Epoch 300/500 | train loss 0.3094 | val loss 3.9146 | val char-PPL 50.13
Epoch 350/500 | train loss 0.3048 | val loss 3.9625 | val char-PPL 52.59
Epoch 400/500 | train loss 0.3019 | val loss 3.9609 | val char-PPL 52.51
Epoch 450/500 | train loss 0.2982 | val loss 3.9386 | val char-PPL 51.35
Epoch 500/500 | train loss 0.2957 | val loss 3.9924 | val char-PPL 54.19
Restored best validation checkpoint (val loss=2.0999).


## 4. Evaluate and generate

Perplexity is measured at the character level. Compare architectures only with the same corpus, split, vocabulary, and broadly comparable settings.

In [4]:

test_loss, test_ppl = evaluate(model, test_loader)
print(f"Test cross-entropy: {test_loss:.4f}")
print(f"Test character-level perplexity: {test_ppl:.2f}")

prompt ="Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden "
print(type(prompt),prompt)
print("\nPrompt:", repr(prompt))
print("\nGenerated text:\n")
print(model.generate(prompt, max_new_chars=350, temperature=0.8))


Test cross-entropy: 2.1115
Test character-level perplexity: 8.26
<class 'str'> Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden 

Prompt: 'Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden '

Generated text:

Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden Sakura Hasuro “(Pate *Saruto -->|Fafhis). **Rand phening ulanke ts ank/Ras, from and cheailizey:** Dumate cole:** Sasuke bicst latesen (kasuke (千鳥) – jivirs sreily sonasuke* Sasuke (春ures:** Sasuke Rally namiya. 

**Firsso Naruto). **Arias:** Kakura Harunaliyerocter by with *Naruto* as and of defearning with Geara) (Parte **Fersed cimattles compias


## 5. Save model artifacts

The checkpoint stores weights and architecture metadata. The JSON file stores vocabulary, hyperparameters, and training history.

In [5]:

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_type": "Vanilla RNN",
    "vocab": itos,
    "block_size": BLOCK_SIZE,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "best_val_loss": best_val_loss,
}
torch.save(checkpoint, "02_rnn_language_model.pt")
with open("02_rnn_metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_type": "Vanilla RNN",
        "vocab": itos,
        "block_size": BLOCK_SIZE,
        "embed_dim": EMBED_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "best_val_loss": best_val_loss,
        "history": history,
    }, f, ensure_ascii=False, indent=2)
print("Saved checkpoint and metadata.")


Saved checkpoint and metadata.


## 6. Experiments

- Increase `HIDDEN_DIM` to 128 if compute allows.
- Change `BLOCK_SIZE` and rerun all cells.
- Train on a larger, varied corpus.
- Compare this model with the other recurrent architecture on the same corpus and split.
- Inspect generations at temperatures 0.5, 0.8, and 1.2.

**Limitations:** this is an educational model, not a production LLM. The built-in demo is too small for reliable conclusions about model quality.